In [5]:
import pandas as pd
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")
STAR_PATH = PROCESSED_PATH / "star_schema"
POWERBI_PATH = PROCESSED_PATH / "powerbi_data"

POWERBI_PATH.mkdir(parents=True, exist_ok=True)

print("STAR_PATH:", STAR_PATH)
print("POWERBI_PATH:", POWERBI_PATH)

STAR_PATH: ../data/processed/star_schema
POWERBI_PATH: ../data/processed/powerbi_data


In [6]:
tables = {
    "DimProduct": pd.read_pickle(STAR_PATH / "dim_product.pkl"),
    "DimCustomer": pd.read_pickle(STAR_PATH / "dim_customer.pkl"),
    "DimSite": pd.read_pickle(STAR_PATH / "dim_site.pkl"),
    "DimPlant": pd.read_pickle(STAR_PATH / "dim_plant.pkl"),
    "DimWeek": pd.read_pickle(STAR_PATH / "dim_week.pkl"),
    "DimDate": pd.read_pickle(STAR_PATH / "dim_date.pkl"),
    "FactSales": pd.read_pickle(STAR_PATH / "fact_sales.pkl"),
    "FactInventory": pd.read_pickle(STAR_PATH / "fact_inventory.pkl"),
}

for name, df in tables.items():
    print(f"{name:15} {df.shape}")

DimProduct      (94868, 26)
DimCustomer     (3406, 23)
DimSite         (3159, 2)
DimPlant        (60, 2)
DimWeek         (86, 7)
DimDate         (335, 11)
FactSales       (831966, 27)
FactInventory   (1367080, 13)


In [7]:
validation = []

for name, df in tables.items():
    validation.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_full_rows": df.duplicated().sum(),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

validation_df = pd.DataFrame(validation)

validation_df

,table,rows,columns,duplicate_full_rows,memory_mb
0,DimProduct,94868,26,0,157.65
1,DimCustomer,3406,23,0,4.54
2,DimSite,3159,2,0,0.22
3,DimPlant,60,2,0,0.00
4,DimWeek,86,7,0,0.01
5,DimDate,335,11,0,0.08
6,FactSales,831966,27,0,618.71
7,FactInventory,1367080,13,0,473.76


In [8]:
dimension_keys = {
    "DimProduct": "product_key",
    "DimCustomer": "customer_key",
    "DimSite": "site_key",
    "DimPlant": "plant_key",
    "DimWeek": "week_key",
    "DimDate": "date_key"
}

pk_validation = []

for table_name, key in dimension_keys.items():
    df = tables[table_name]

    pk_validation.append({
        "table": table_name,
        "primary_key": key,
        "rows": len(df),
        "unique_keys": df[key].nunique(),
        "duplicate_keys": df.duplicated(key).sum(),
        "missing_keys": df[key].isna().sum()
    })

pk_validation_df = pd.DataFrame(pk_validation)

pk_validation_df

,table,primary_key,rows,unique_keys,duplicate_keys,missing_keys
0,DimProduct,product_key,94868,94868,0,0
1,DimCustomer,customer_key,3406,3406,0,0
2,DimSite,site_key,3159,3159,0,0
3,DimPlant,plant_key,60,60,0,0
4,DimWeek,week_key,86,86,0,0
5,DimDate,date_key,335,335,0,0


In [9]:
fact_validation = pd.DataFrame({
    "check": [
        "FactSales rows",
        "FactInventory rows",
        "FactSales unique ID",
        "FactInventory unique ID",
        "FactSales unknown product",
        "FactSales unknown customer",
        "FactInventory unknown product"
    ],
    "value": [
        len(tables["FactSales"]),
        len(tables["FactInventory"]),
        tables["FactSales"]["sales_fact_id"].nunique(),
        tables["FactInventory"]["inventory_fact_id"].nunique(),
        (tables["FactSales"]["product_key"] == 0).sum(),
        (tables["FactSales"]["customer_key"] == 0).sum(),
        (tables["FactInventory"]["product_key"] == 0).sum()
    ]
})

fact_validation

,check,value
0,FactSales rows,831966
1,FactInventory rows,1367080
2,FactSales unique ID,831966
3,FactInventory unique ID,1367080
4,FactSales unknown product,6
5,FactSales unknown customer,1
6,FactInventory unknown product,29


In [10]:
for name, df in tables.items():
    output_file = POWERBI_PATH / f"{name}.csv"

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    size_mb = output_file.stat().st_size / 1024**2

    print(
        f"{name:15} -> {output_file.name:25} "
        f"{size_mb:.2f} MB"
    )

DimProduct      -> DimProduct.csv            26.92 MB
DimCustomer     -> DimCustomer.csv           0.56 MB
DimSite         -> DimSite.csv               0.04 MB
DimPlant        -> DimPlant.csv              0.00 MB
DimWeek         -> DimWeek.csv               0.00 MB
DimDate         -> DimDate.csv               0.02 MB
FactSales       -> FactSales.csv             174.28 MB
FactInventory   -> FactInventory.csv         181.63 MB


In [11]:
csv_files = sorted(POWERBI_PATH.glob("*.csv"))

for file in csv_files:
    print(
        file.name,
        "->",
        round(file.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nTotal CSV files:", len(csv_files))

DimCustomer.csv -> 0.56 MB
DimDate.csv -> 0.02 MB
DimPlant.csv -> 0.0 MB
DimProduct.csv -> 26.92 MB
DimSite.csv -> 0.04 MB
DimWeek.csv -> 0.0 MB
FactInventory.csv -> 181.63 MB
FactSales.csv -> 174.28 MB

Total CSV files: 8


In [12]:
csv_check = []

for name in tables.keys():
    file = POWERBI_PATH / f"{name}.csv"

    df_check = pd.read_csv(
        file,
        low_memory=False
    )

    csv_check.append({
        "table": name,
        "before_rows": len(tables[name]),
        "csv_rows": len(df_check),
        "difference": len(df_check) - len(tables[name])
    })

csv_check_df = pd.DataFrame(csv_check)

csv_check_df

,table,before_rows,csv_rows,difference
0,DimProduct,94868,94868,0
1,DimCustomer,3406,3406,0
2,DimSite,3159,3159,0
3,DimPlant,60,60,0
4,DimWeek,86,86,0
5,DimDate,335,335,0
6,FactSales,831966,831966,0
7,FactInventory,1367080,1367080,0


In [13]:
fact_sales_csv = pd.read_csv(
    POWERBI_PATH / "FactSales.csv",
    low_memory=False
)

fact_inventory_csv = pd.read_csv(
    POWERBI_PATH / "FactInventory.csv",
    low_memory=False
)

final_validation = pd.DataFrame({
    "metric": [
        "FactSales rows",
        "FactSales sold_quantity",
        "FactSales cost_price",
        "FactSales net_price",
        "FactInventory rows",
        "FactInventory quantity",
        "FactInventory total_amount"
    ],

    "expected": [
        len(tables["FactSales"]),
        tables["FactSales"]["sold_quantity"].sum(),
        tables["FactSales"]["cost_price"].sum(),
        tables["FactSales"]["net_price"].sum(),
        len(tables["FactInventory"]),
        tables["FactInventory"]["quantity"].sum(),
        tables["FactInventory"]["total_amount"].sum()
    ],

    "csv": [
        len(fact_sales_csv),
        fact_sales_csv["sold_quantity"].sum(),
        fact_sales_csv["cost_price"].sum(),
        fact_sales_csv["net_price"].sum(),
        len(fact_inventory_csv),
        fact_inventory_csv["quantity"].sum(),
        fact_inventory_csv["total_amount"].sum()
    ]
})

final_validation["difference"] = (
    final_validation["csv"] -
    final_validation["expected"]
)

final_validation

,metric,expected,csv,difference
0,FactSales rows,831966,831966,0
1,FactSales sold_quantity,1174515,1174515,0
2,FactSales cost_price,254170428358,254170428358,0
3,FactSales net_price,332230025498,332230025498,0
4,FactInventory rows,1367080,1367080,0
5,FactInventory quantity,2253264,2253264,0
6,FactInventory total_amount,81552273459,81552273459,0
